In [1]:
from google.colab import drive
drive.mount('/content/drive')


import torch
import numpy as np
import pandas as pd
try:
  from kan import KAN
except:
  !pip3 install pykan
  from kan import KAN

import os
import sys
print(os.getcwd())
os.chdir('/content/drive/MyDrive/codes_kan_covnet')
print(os.getcwd())
sys.path.insert(1, os.path.join(os.getcwd(), "Datagen"))
sys.path.insert(1, os.path.join(os.getcwd(), 'NN_build'))
sys.path.insert(1, os.path.join(os.getcwd(), 'Covnet_source_codes'))
sys.path.insert(1, os.path.join(os.getcwd(), 'two_D'))


from NN_build import cov_networks as CN_KAN


from two_D import datagen_nonseparable2 as dnsep
import CovNetworks as CN
from Important_functions import loss_COV, batch_CV, cnet_optim_best

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.1/78.1 kB 5.6 MB/s eta 0:00:00
/content
/content/drive/MyDrive/codes_kan_covnet


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Emipirical and BESTSEP

In [2]:

def empirical_relative_error(folder, repl=1):
    x_path = os.path.join(folder, f"Example{repl}.dat")
    idx_path = os.path.join(folder, f"indices{repl}.dat")
    true_cov_path = os.path.join(folder, f"True_cov{repl}.dat")

    X = np.loadtxt(x_path)              # shape: (N, D)
    idx = np.loadtxt(idx_path, dtype=int)  # shape: (2, M)
    true_cov = np.loadtxt(true_cov_path)   # shape: (M,)

    idx1 = idx[0, :]
    idx2 = idx[1, :]

    # empirical covariance at M test pairs
    pred_cov = np.mean(X[:, idx1] * X[:, idx2], axis=0)

    mse = np.mean((pred_cov - true_cov) ** 2)
    rel = np.linalg.norm(pred_cov - true_cov) / np.linalg.norm(true_cov)

    return mse, rel, pred_cov

In [3]:


############################################################ FROM TRUE COV

# def rotation_matrix_2d(theta=np.pi / 4):
#     return np.array([
#         [np.cos(theta), -np.sin(theta)],
#         [np.sin(theta),  np.cos(theta)]
#     ])


# def separable_predict(true_locations, cov_fn, rotation=False, theta=np.pi / 4):
#     d = true_locations.shape[1] // 2

#     u = true_locations[:, :d]
#     v = true_locations[:, d:]

#     if rotation:
#         if d != 2:
#             raise ValueError("rotation=True is currently supported only for d=2.")

#         O = rotation_matrix_2d(theta)
#         u = u @ O.T
#         v = v @ O.T

#     pred = np.zeros(u.shape[0])

#     for i in range(u.shape[0]):
#         cov = 1.0
#         for j in range(d):
#             cov *= cov_fn(u[i, j], v[i, j])
#         pred[i] = cov

#     return pred


# def separable_relative_error(folder, cov_fn, rotation = False, theta = np.pi/4, repl = 1):
#     true_loc_path = os.path.join(folder, f"True_locations{repl}.dat")
#     true_cov_path = os.path.join(folder, f"True_cov{repl}.dat")

#     true_locations = np.loadtxt(true_loc_path)
#     true_cov = np.loadtxt(true_cov_path)

#     pred_cov = separable_predict(true_locations, cov_fn, rotation = rotation, theta = theta)

#     mse = np.mean((pred_cov - true_cov) ** 2)
#     rel = np.linalg.norm(pred_cov - true_cov) / np.linalg.norm(true_cov)

#     return mse, rel, pred_cov






def best_separable_fit_2d(X, K, center=True):
    """
    Estimate best separable covariance from data.

    X shape: (N, K^2)
    Returns:
        A_hat shape: (K, K)
        B_hat shape: (K, K)

    Approximation:
        Cov((i,j),(k,l)) ≈ A_hat[i,k] * B_hat[j,l]
    """

    if center:
        X = X - X.mean(axis=0, keepdims=True)

    N, D = X.shape
    assert D == K**2, "For d=2, X must have K^2 columns."

    # empirical covariance, shape (K^2, K^2)
    S = (X.T @ X) / N

    # rearrange covariance into matrix for rank-1 Kronecker approximation
    Rmat = np.zeros((K*K, K*K))

    for i in range(K):
        for k in range(K):
            row = i * K + k

            for j in range(K):
                for l in range(K):
                    col = j * K + l

                    idx1 = i * K + j   # location (i,j)
                    idx2 = k * K + l   # location (k,l)

                    Rmat[row, col] = S[idx1, idx2]

    # best rank-1 approximation using SVD
    U, s, Vt = np.linalg.svd(Rmat, full_matrices=False)

    sigma = s[0]
    u1 = U[:, 0]
    v1 = Vt[0, :]

    A_hat = np.sqrt(sigma) * u1.reshape(K, K)
    B_hat = np.sqrt(sigma) * v1.reshape(K, K)

    # fix sign if needed
    if np.mean(A_hat) < 0:
        A_hat = -A_hat
        B_hat = -B_hat

    # symmetrize
    A_hat = 0.5 * (A_hat + A_hat.T)
    B_hat = 0.5 * (B_hat + B_hat.T)

    return A_hat, B_hat




def bestsep_relative_error(folder, repl=1, K=5, center=True):
    x_path = os.path.join(folder, f"Example{repl}.dat")
    idx_path = os.path.join(folder, f"indices{repl}.dat")
    true_cov_path = os.path.join(folder, f"True_cov{repl}.dat")

    X = np.loadtxt(x_path)
    X = np.atleast_2d(X)   # important fix

    idx = np.loadtxt(idx_path, dtype=int)
    true_cov = np.loadtxt(true_cov_path)

    print("Loaded X shape:", X.shape)
    print("Expected columns:", K**2)

    if X.shape[1] != K**2:
        raise ValueError(
            f"X has {X.shape[1]} columns, but K={K} requires K^2={K**2}. "
            f"Check whether you generated data with a different K."
        )

    A_hat, B_hat = best_separable_fit_2d(X, K, center=center)

    idx1 = idx[0, :]
    idx2 = idx[1, :]

    i1 = idx1 // K
    j1 = idx1 % K

    i2 = idx2 // K
    j2 = idx2 % K

    pred_cov = A_hat[i1, i2] * B_hat[j1, j2]

    mse = np.mean((pred_cov - true_cov) ** 2)
    rel = np.linalg.norm(pred_cov - true_cov) / np.linalg.norm(true_cov)

    return mse, rel, pred_cov

# Experiment

## Simulation

In [ ]:
N = 500                 # random surface (location)
K = 25                  # Observed points in each surface
d = 2                   # Dimensions
replicates = 20          # replication
M = 50000                # Number of observation to be tested as test_set

### BM

In [ ]:


# BM without rotation
dnsep.datagen_and_print(
    N=N,
    K=K,
    d=d,
    replicates=replicates,
    method = lambda s, t: dnsep.BM(s, t),
    cov_name = 'BM',
    rotation=False,
    M=M,
    base_dir="Simulation",
    seed=np.random.randint(100)
)

Saving files to: Simulation/BM
Basis generation started
Basis generated
Full covariance shape: (625, 625)
Replicate 1
Replicate 2
Replicate 3
Replicate 4
Replicate 5
Replicate 6
Replicate 7
Replicate 8
Replicate 9
Replicate 10
Replicate 11
Replicate 12
Replicate 13
Replicate 14
Replicate 15
Replicate 16
Replicate 17
Replicate 18
Replicate 19
Replicate 20
Data generation completed.


In [ ]:
# BM with rotation
dnsep.datagen_and_print(
    N=N,
    K=K,
    d=d,
    replicates=replicates,
    method = lambda s, t: dnsep.BM(s, t),
    cov_name = 'BM',
    rotation=True,
    M=M,
    base_dir="Simulation",
    seed=np.random.randint(100)
)



Saving files to: Simulation/BM_rotated
Basis generation started
Basis generated
Full covariance shape: (625, 625)
Replicate 1
Replicate 2
Replicate 3
Replicate 4
Replicate 5
Replicate 6
Replicate 7
Replicate 8
Replicate 9
Replicate 10
Data generation completed.


### Integrated BM (IBM)

In [ ]:
# iBM without rotation
dnsep.datagen_and_print(
    N=N,
    K=K,
    d=d,
    replicates=replicates,
    method=lambda s, t: dnsep.iBM(s, t),
    cov_name="iBM",
    rotation=False,
    theta=np.pi / 4,
    M=M,
    base_dir="Simulation",
    seed=np.random.randint(100)
)


Saving files to: Simulation/iBM
Basis generation started
Basis generated
Full covariance shape: (625, 625)
Replicate 1
Replicate 2
Replicate 3
Data generation completed.


In [ ]:
# iBM with 45-degree rotation
dnsep.datagen_and_print(
    N=N,
    K=K,
    d=d,
    replicates=replicates,
    method=lambda s, t: dnsep.iBM(s, t),
    cov_name="iBM",
    rotation=True,
    theta=np.pi / 4,
    M=M,
    base_dir="Simulation",
    seed=np.random.randint(100)
)

Saving files to: Simulation/iBM_rotated
Basis generation started
Basis generated
Full covariance shape: (625, 625)
Replicate 1
Replicate 2
Replicate 3
Data generation completed.


### Matern

In [ ]:
#Matern nu=0.001
dnsep.datagen_and_print(
    N=N,
    K=K,
    d=d,
    replicates=replicates,
    method=lambda s, t: dnsep.matern(s, t, nu=0.001),
    cov_name="Matern_.001",
    rotation=False,
    M=M,
    base_dir="Simulation",
    seed=np.random.randint(100)
)

Saving files to: Simulation/Matern_.001
Basis generation started
Basis generated
Full covariance shape: (625, 625)
Replicate 1
Replicate 2
Replicate 3
Data generation completed.


In [ ]:
# Matern nu=0.01
dnsep.datagen_and_print(
    N=N,
    K=K,
    d=d,
    replicates=replicates,
    method=lambda s, t: dnsep.matern(s, t, nu=0.01),
    cov_name="Matern_.01",
    rotation=False,
    M=M,
    base_dir="Simulation",
    seed=np.random.randint(100)
)

Saving files to: Simulation/Matern_.01
Basis generation started
Basis generated
Full covariance shape: (625, 625)
Replicate 1
Replicate 2
Replicate 3
Data generation completed.


In [ ]:
# Matern nu=0.1
dnsep.datagen_and_print(
    N=N,
    K=K,
    d=d,
    replicates=replicates,
    method=lambda s, t: dnsep.matern(s, t, nu=0.1),
    cov_name="Matern_.1",
    rotation=False,
    M=M,
    base_dir="Simulation",
    seed=np.random.randint(100)
)

Saving files to: Simulation/Matern_.1
Basis generation started
Basis generated
Full covariance shape: (625, 625)
Replicate 1
Replicate 2
Replicate 3
Data generation completed.


In [ ]:
# Matern nu=1
dnsep.datagen_and_print(
    N=N,
    K=K,
    d=d,
    replicates=replicates,
    method=lambda s, t: dnsep.matern(s, t, nu=1),
    cov_name="Matern_1.0",
    rotation=False,
    M=M,
    base_dir="Simulation",
    seed=np.random.randint(100)
)

Saving files to: Simulation/Matern_1.0
Basis generation started
Basis generated
Full covariance shape: (625, 625)
Replicate 1
Replicate 2
Replicate 3
Data generation completed.


## Data Load

In [4]:
import os
import numpy as np
import torch
import torch.optim as optim


def load_covnet_replicate(folder, repl=1, device="cpu"):
    loc_path = os.path.join(folder, f"locations{repl}.dat")
    x_path = os.path.join(folder, f"Example{repl}.dat")
    true_loc_path = os.path.join(folder, f"True_locations{repl}.dat")
    true_cov_path = os.path.join(folder, f"True_cov{repl}.dat")

    u = np.loadtxt(loc_path)              # (D, d)
    x = np.loadtxt(x_path)                # (N, D)
    true_locs = np.loadtxt(true_loc_path) # (M, 2d)
    true_cov = np.loadtxt(true_cov_path)  # (M,)

    d = u.shape[1]

    u_test = true_locs[:, :d]
    v_test = true_locs[:, d:]

    return {
        "u": torch.tensor(u, dtype=torch.float32, device=device),
        "x": torch.tensor(x, dtype=torch.float32, device=device),
        "u_test": torch.tensor(u_test, dtype=torch.float32, device=device),
        "v_test": torch.tensor(v_test, dtype=torch.float32, device=device),
        "true_cov": torch.tensor(true_cov, dtype=torch.float32, device=device),
    }




## Loss function

In [5]:
def covnet_loss(model, u, x):
   # x = x - x.mean(dim=0, keepdim=True)

    x_hat = model(u)
   # x_hat = x_hat - x_hat.mean(dim=0, keepdim=True)

    D = x.shape[1]

    l1 = (x @ x.T) / D
    l2 = (x_hat @ x_hat.T) / D
    l3 = (x @ x_hat.T) / D

    loss = (
        torch.mean(l1**2)
        + torch.mean(l2**2)
        - 2 * torch.mean(l3**2)
        + torch.mean(l1)**2
        + torch.mean(l2)**2
        - 2 * torch.mean(l3)**2
    )

    return loss


def test_on_M_locations(model, u_test, v_test, true_cov):
    model.eval()
    device = next(model.parameters()).device

    u_test = u_test.to(device)
    v_test = v_test.to(device)
    true_cov = true_cov.to(device)

    with torch.no_grad():
        x_u = model(u_test)  # (N, M)
        x_v = model(v_test)  # (N, M)

      #  x_u_c = x_u - x_u.mean(dim=0, keepdim=True)
      #  x_v_c = x_v - x_v.mean(dim=0, keepdim=True)

        pred_cov = torch.mean(x_u * x_v, dim=0)

        mse = torch.mean((pred_cov - true_cov) ** 2)
        rel_l2 = torch.norm(pred_cov - true_cov) / torch.norm(true_cov)

    return mse.item(), rel_l2.item(), pred_cov




def debug_cov_scale(model, data):
    with torch.no_grad():
        xu = model(data["u_test"])
        xv = model(data["v_test"])

        pred_cov = torch.mean(xu * xv, dim=0)

        print("true mean:", data["true_cov"].mean().item())
        print("true std:", data["true_cov"].std().item())
        print("pred mean:", pred_cov.mean().item())
        print("pred std:", pred_cov.std().item())
        print("rel:", (torch.norm(pred_cov - data["true_cov"]) / torch.norm(data["true_cov"])).item())







from sklearn.model_selection import KFold
import pandas as pd


def cv_loss_against_validation_cov(model, u, x_val):
    """
    CV loss: compare fitted covariance from model with validation empirical covariance.
    x_val shape: (N_val, D)
    model(u) shape: (N_train, D) or (N_model, D)
    """
    model.eval()
    with torch.no_grad():
        x_hat = model(u)

        C_val = (x_val.T @ x_val) / x_val.shape[0]
        C_hat = (x_hat.T @ x_hat) / x_hat.shape[0]

        return torch.mean((C_hat - C_val) ** 2).item()




## Training

In [ ]:

def train_one_replicate(
    folder,
    repl=1,
    d=2,
    N=500,
    R=30,
    depth = 3,
    hidden_dim = 3,
    epochs=100,
    lr=1e-2,
    batch_size = 512,
    type = 'KAN',
    device="cpu"
):


    if type == 'Shallow_KAN':
        model = CN_KAN.CovNetShallowKAN(
            d=d,
            N=N,
            R=R,
            grid=3,
            k=3
        ).to(device)

    elif type == 'Shallow_net':
        model = CN.CovNetShallow(
            d=d,
            N=N,
            R=R,
            act_fn=torch.nn.Sigmoid(),
            init = torch.nn.init.xavier_normal_
        ).to(device)




    elif type == 'Deep_KAN':
        model = CN_KAN.CovNetDeepKAN(
            d=d,
            N=N,
            R=R,
            depth=depth,
            hidden_dim=hidden_dim,
            grid=3,
            k=3,
            seed=np.random.randint(500)
        ).to(device)

    elif type == 'Deep_net':
        model = CN.CovNetDeep(
            d = d,
            N = N,
            R = R,
            depth = depth,
            n_nodes = hidden_dim,
            act_fn = torch.nn.Sigmoid(),
            init = torch.nn.init.xavier_normal_).to(device)







    elif type == 'DeepShared_KAN':
        model = CN_KAN.CovNetDeepSharedKAN(
            d=d,
            N=N,
            R=R,
            depth=depth,
            hidden_dim=hidden_dim,
            grid=3,
            k=3,
            seed=np.random.randint(500)
        ).to(device)

    elif type == 'DeepShared_net':
        model = CN.CovNetDeepShared(
            d = d,
            N = N,
            R = R,
            depth = depth,
            act_fn = torch.nn.Sigmoid(),
            init = torch.nn.init.xavier_normal_).to(device)



    else:
        raise ValueError(
            f"Unknown model type: {type}. "
            "Choose one of: Shallow_KAN, Shallow_net, Deep_KAN, "
            "Deep_net, DeepShared_KAN or DeepShared_net."
        )


    optimizer = optim.Adam(model.params , lr=lr)
    data = load_covnet_replicate(folder, repl, device=device)
    debug_cov_scale(model=model, data=data)

    u_full = data["u"]
    x_full = data["x"]

    D = u_full.shape[0]


    for epoch in range(epochs):
        model.train()

        perm = torch.randperm(D, device=device)
        epoch_loss = 0.0
        n_batches = 0

        for start in range(0, D, batch_size):
            idx = perm[start:start + batch_size]

            u_batch = u_full[idx, :]
            x_batch = x_full[:, idx]

            optimizer.zero_grad()

            x_hat = model(u_batch)
            loss = loss_COV(x_batch, x_hat)

            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()
            n_batches += 1

        if epoch % 10 == 0:
            test_mse, test_rel, _ = test_on_M_locations(
                model,
                data["u_test"],
                data["v_test"],
                data["true_cov"]
            )

            print(
                f"Rep {repl} | Epoch {epoch} | "
                f"loss={epoch_loss:.8f} | "
                f"test_mse={test_mse:.8f} | "
                f"test_rel={test_rel:.6f}"
            )



    final_mse, final_rel, pred_cov = test_on_M_locations(
        model,
        data["u_test"],
        data["v_test"],
        data["true_cov"]
    )

    return model, final_mse, final_rel

In [ ]:
N = 100
d = 2
M = 50000


dnsep.datagen_and_print(
        N=N,
        K=5,
        d=d,
        replicates=1,
        method = lambda s, t: dnsep.BM(s, t),
        cov_name = 'BM',
        rotation=True,
        M=M,
        base_dir="Simulation",
        seed=np.random.randint(100)
    )

for i in [20]:
    print(f'\n################### R = {i}')


    folder = "Simulation/BM_rotated"



    model, mse, rel = train_one_replicate(
        folder=folder,
        repl=1,
        d=2,
        N=100,
        R=i,
        epochs=100,
        lr=0.01,
        device="cpu",
        type = 'Shallow_net'
    )


    print("\nFinal MSE:", mse)
    print("Final relative L2 error:", rel)


    emp_mse, emp_rel, _ = empirical_relative_error(folder, repl = 1)
    sep_mse, sep_rel, _ = bestsep_relative_error(folder, repl = 1, K = 5, center = True)

    print("empirical:", emp_rel)
    print("separable:", sep_rel, '\n')

Saving files to: Simulation/BM_rotated
Basis generation started
Basis generated
Full covariance shape: (25, 25)
Replicate 1
Data generation completed.

################### R = 20
true mean: 0.03977075591683388
true std: 0.06783795356750488
pred mean: 0.07913808524608612
pred std: 0.0004872485005762428
rel: 0.9968746304512024
Rep 1 | Epoch 0 | loss=0.00681493 | test_mse=0.00461583 | test_rel=0.863981
Rep 1 | Epoch 10 | loss=0.00555783 | test_mse=0.00459008 | test_rel=0.861567
Rep 1 | Epoch 20 | loss=0.00547625 | test_mse=0.00461081 | test_rel=0.863511
Rep 1 | Epoch 30 | loss=0.00545295 | test_mse=0.00459382 | test_rel=0.861918
Rep 1 | Epoch 40 | loss=0.00541415 | test_mse=0.00454088 | test_rel=0.856938
Rep 1 | Epoch 50 | loss=0.00537636 | test_mse=0.00451644 | test_rel=0.854628
Rep 1 | Epoch 60 | loss=0.00533282 | test_mse=0.00451210 | test_rel=0.854217
Rep 1 | Epoch 70 | loss=0.00528014 | test_mse=0.00448437 | test_rel=0.851588
Rep 1 | Epoch 80 | loss=0.00521018 | test_mse=0.00446546 |

In [ ]:
results = []

for repl in range(1, replicates + 1):
    model, mse, rel = train_one_replicate(
        folder="Simulation/iBM",
        repl=repl,
        d=2,
        N=100,
        R=30,
        epochs=2000,
        lr=1e-3,
        device="cpu",
        type='CN'
    )

    results.append((repl, mse, rel))

print(np.mean(results, axis=0)[2], np.std(results, axis=0)[2])


ValueError: Unknown model type: CN. Choose one of: Shallow_KAN, Shallow_net, Deep_KAN, Deep_net, DeepShared_KAN or DeepShared_net.

# Full Experiment

## Helper

In [19]:
def build_CN_model(
    model_type,
    d,
    N,
    R,
    depth=3,
    hidden_dim=None,
    grid = 2,
    k = 1,
    device="cuda"
):
    if hidden_dim is None:
        hidden_dim = R

    if model_type == "Shallow_net":
        model = CN.CovNetShallow(
            d=d,
            N=N,
            R=R,
            act_fn=torch.nn.Sigmoid(),
            init=torch.nn.init.xavier_normal_
        ).to(device)

    elif model_type == "Deep_net":
        model = CN.CovNetDeep(
            d=d,
            N=N,
            R=R,
            depth=depth,
            n_nodes=hidden_dim,
            act_fn=torch.nn.Sigmoid(),
            init=torch.nn.init.xavier_normal_
        ).to(device)

    elif model_type == "DeepShared_net":
        model = CN.CovNetDeepShared(
            d=d,
            N=N,
            R=R,
            depth=depth,
            act_fn=torch.nn.Sigmoid(),
            init=torch.nn.init.xavier_normal_
        ).to(device)

    else:
        raise ValueError(
            f"Unknown CN model_type: {model_type}. "
            "Use Shallow_net, Deep_net, or DeepShared_net."
        )




    return model


def build_KAN_model(
    model_type,
    d,
    N,
    R,
    depth=3,
    hidden_dim=None,
    grid=3,
    k=3,
    seed=None,
    device="cuda"
):
    if hidden_dim is None:
        hidden_dim = R

    if seed is None:
        seed = np.random.randint(5000)

    if model_type == "Shallow_KAN":
        model = CN_KAN.CovNetShallowKAN(
            d=d,
            N=N,
            R=R,
            grid=grid,
            k=k,
            seed=seed
        ).to(device)

    elif model_type == "Deep_KAN":
        model = CN_KAN.CovNetDeepKAN(
            d=d,
            N=N,
            R=R,
            depth=depth,
            hidden_dim=hidden_dim,
            grid=grid,
            k=k,
            seed=seed
        ).to(device)

    elif model_type == "DeepShared_KAN":
        model = CN_KAN.CovNetDeepSharedKAN(
            d=d,
            N=N,
            R=R,
            depth=depth,
            hidden_dim=hidden_dim,
            grid=grid,
            k=k,
            seed=seed
        ).to(device)

    else:
        raise ValueError(
            f"Unknown KAN model_type: {model_type}. "
            "Use Shallow_KAN, Deep_KAN, or DeepShared_KAN."
        )

    return model



def train_model(model, u_train, x_train, epochs=500, lr=1e-2, batch_size=512):

    device = next(model.parameters()).device

    # Ensure model parameters are explicitly on the correct device
    model.to(device)

    u_train = u_train.to(device)
    x_train = x_train.to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    D = u_train.shape[0]

    # print("model device:", next(model.parameters()).device)
    # print("u device:", u_train.device)
    # print("x device:", x_train.device)

    for epoch in range(epochs):
        model.train()

        perm = torch.randperm(D, device=device)

        for start in range(0, D, batch_size):
            idx = perm[start:start + batch_size]

            u_batch = u_train[idx, :]
            x_batch = x_train[:, idx]

            optimizer.zero_grad()
            x_hat = model(u_batch)

            loss = loss_COV(x_batch, x_hat)

            loss.backward()
            optimizer.step()

    return model








def fivefold_cv_select(
    model_type,
    data,
    candidate_grid,
    build_model_fn,
    train_model_fn,
    epochs,
    lr,
    batch_size,
    device="cuda"
):
    """
    CV split is over observations/fields, i.e. rows of x.
    """
    u = data["u"]
    x = data["x"]
    N = x.shape[0]
    d = u.shape[1]

    kf = KFold(n_splits=2, shuffle=True, random_state=123)

    cv_records = []

    for cand in candidate_grid:
        fold_scores = []

        for train_ids, val_ids in kf.split(np.arange(N)):
            train_ids = torch.tensor(train_ids, dtype=torch.long, device=device)
            val_ids = torch.tensor(val_ids, dtype=torch.long, device=device)

            x_train = x[train_ids, :]
            x_val = x[val_ids, :]

            model = build_model_fn(
                model_type=model_type,
                d=d,
                N=x_train.shape[0],   # important: model output matches training fields
                R=cand["R"],
                depth=cand.get("depth", 1),
                hidden_dim=cand.get("hidden_dim", cand["R"]),
                device=device
            )

            model = train_model_fn(
                model=model,
                u_train=u,
                x_train=x_train,
                epochs=epochs,
                lr=lr,
                batch_size=batch_size
            )

            score = cv_loss_against_validation_cov(model, u, x_val)
            fold_scores.append(score)

        cv_records.append({
            **cand,
            "cv_score": float(np.mean(fold_scores))
        })

    cv_df = pd.DataFrame(cv_records)
    best = cv_df.loc[cv_df["cv_score"].idxmin()].to_dict()

    return best, cv_df

In [7]:
def run_one_model_table1_style(
    model_type,
    data,
    build_model_fn,
    train_model_fn,
    candidate_grid,
    epochs_cv=2000,
    epochs_final=5000,
    lr=1e-2,
    batch_size=512,
    device="cuda"
):
    # 1. CV-selected hyperparameter
    best_params, cv_df = fivefold_cv_select(
        model_type=model_type,
        data=data,
        candidate_grid=candidate_grid,
        build_model_fn=build_model_fn,
        train_model_fn=train_model_fn,
        epochs=epochs_cv,
        lr=lr,
        batch_size=batch_size,
        device=device
    )

    # 2. Refit selected model on all N fields
    u = data["u"]
    x = data["x"]
    d = u.shape[1]
    N_full = x.shape[0]

    selected_model = build_model_fn(
        model_type=model_type,
        d=d,
        N=N_full,
        R=int(best_params["R"]),
        depth=int(best_params.get("depth", 1)),
        hidden_dim=int(best_params.get("hidden_dim", best_params["R"])),
        device=device
    )

    selected_model = train_model_fn(
        model=selected_model,
        u_train=u,
        x_train=x,
        epochs=epochs_final,
        lr=lr,
        batch_size=batch_size
    )

    selected_mse, selected_rel, _ = test_on_M_locations(
        selected_model,
        data["u_test"],
        data["v_test"],
        data["true_cov"]
    )

    # 3. Oracle-best over hyperparameter grid
    oracle_rels = []

    for cand in candidate_grid:
        model = build_model_fn(
            model_type=model_type,
            d=d,
            N=N_full,
            R=cand["R"],
            depth=cand.get("depth", 1),
            hidden_dim=cand.get("hidden_dim", cand["R"]),
            device=device
        )

        model = train_model_fn(
            model=model,
            u_train=u,
            x_train=x,
            epochs=epochs_final,
            lr=lr,
            batch_size=batch_size
        )

        _, rel, _ = test_on_M_locations(
            model,
            data["u_test"],
            data["v_test"],
            data["true_cov"]
        )

        oracle_rels.append(rel)

    oracle_best_rel = min(oracle_rels)
    excess = selected_rel - oracle_best_rel

    return {
        "selected_rel": selected_rel,
        "selected_mse": selected_mse,
        "excess": excess,
        "best_params": best_params,
        "cv_df": cv_df
    }

In [27]:
CN_SHALLOW_GRID = [
    {"R": R}
    for R in [10]
]

CN_DEEP_GRID = [
    {"R": R, "depth": L, "hidden_dim": R}
    for R in [5, 10, 20, 40]
    for L in [2, 3, 4]
]



KAN_SHALLOW_GRID = [
    {"R": R, "grid": grid, "k": 3}
    for R in [10]
    for grid in [3]
]

KAN_DEEP_GRID = [
    {"R": R, "depth": L, "hidden_dim": R, "grid": grid, "k": 3}
    for R in [5, 10, 20, 40]
    for L in [3, 4, 5]
    for grid in [3, 5]
]


In [ ]:
folder = 'Simulation/BM'
device = 'cuda'
all_rows = []

for repl in range(1, 21):
    data = load_covnet_replicate(folder, repl=repl, device=device)

    emp_mse, emp_rel, _ = empirical_relative_error(folder, repl=repl)
    sep_mse, sep_rel, _ = bestsep_relative_error(folder, repl=repl, K=25)

    all_rows.append({"replicate": repl, "method": "empirical", "rel": emp_rel, "excess": np.nan})
    all_rows.append({"replicate": repl, "method": "BestSep", "rel": sep_rel, "excess": np.nan})

    for model_type, grid in [
        ("Shallow_KAN", KAN_SHALLOW_GRID),
  #      ("Deep_KAN", KAN_DEEP_GRID),
   #     ("DeepShared_KAN", KAN_DEEP_GRID),
    ]:
        out = run_one_model_table1_style(
            model_type=model_type,
            data=data,
            build_model_fn=build_KAN_model,
            train_model_fn=train_model,
            candidate_grid=grid,
            epochs_cv=200,
            epochs_final=300,
            lr=1e-2,
            batch_size=512,
            device=device
        )

        all_rows.append({
            "replicate": repl,
            "method": model_type,
            "rel": out["selected_rel"],
            "excess": out["excess"]
        })

        print(f'{model_type} done!!!')

    pd.DataFrame(all_rows)#.to_csv("table1_running_results.csv", index=False)

In [12]:
import numpy as np
import pandas as pd
import torch


def validation_cov_loss(model, u, x_val):
    """
    Compare model covariance to validation empirical covariance.
    x_val: (N_val, D)
    """
    model.eval()
    with torch.no_grad():
        x_hat = model(u)  # (N_model, D)

        C_val = (x_val.T @ x_val) / x_val.shape[0]
        C_hat = (x_hat.T @ x_hat) / x_hat.shape[0]

        loss = torch.mean((C_hat - C_val) ** 2)

    return loss.item()


def train_validation_tune_one_model(
    model_type,
    data,
    candidate_grid,
    build_fn,
    train_fn,
    train_ratio=0.8,
    epochs_tune=1000,
    epochs_final=5000,
    lr=1e-2,
    batch_size=512,
    device="cuda",
    seed=123
):
    """
    1. Split fields: train/validation
    2. Train each hyperparameter on train fields
    3. Select lowest validation covariance loss
    4. Refit selected hyperparameter on all fields
    5. Test on M random pairs
    """

    torch.manual_seed(seed)
    np.random.seed(seed)

    u = data["u"].to(device)
    x = data["x"].to(device)

    N = x.shape[0]
    d = u.shape[1]

    # -------------------------------------------------
    # 1. Split fields, not locations
    # -------------------------------------------------
    perm = torch.randperm(N, device=device)
    n_train = int(train_ratio * N)

    train_ids = perm[:n_train]
    val_ids = perm[n_train:]

    x_train = x[train_ids, :]
    x_val = x[val_ids, :]

    tuning_records = []

    # -------------------------------------------------
    # 2. Grid search using validation covariance loss
    # -------------------------------------------------
    for cand in candidate_grid:
        print(f"\nTuning {model_type}: {cand}")

        model = build_fn(
            model_type=model_type,
            d=d,
            N=x_train.shape[0],          # important: model output matches train fields
            R=cand["R"],
            depth=cand.get("depth", 1),
            hidden_dim=cand.get("hidden_dim", cand["R"]),
            grid=cand.get("grid", 3),
            k=cand.get("k", 3),
            device=device
        )

        model = train_fn(
            model=model,
            u_train=u,
            x_train=x_train,
            epochs=epochs_tune,
            lr=lr,
            batch_size=batch_size
        )

        val_loss = validation_cov_loss(model, u, x_val)

        tuning_records.append({
            **cand,
            "val_loss": val_loss
        })

        print(f"Validation loss = {val_loss:.8e}")

    tuning_df = pd.DataFrame(tuning_records)
    best_params = tuning_df.loc[tuning_df["val_loss"].idxmin()].to_dict()

    print("\nBest params:", best_params)

    # -------------------------------------------------
    # 3. Refit selected model on all N fields
    # -------------------------------------------------
    final_model = build_fn(
        model_type=model_type,
        d=d,
        N=N,
        R=int(best_params["R"]),
        depth=int(best_params.get("depth", 1)),
        hidden_dim=int(best_params.get("hidden_dim", best_params["R"])),
        grid=int(best_params.get("grid", 3)),
        k=int(best_params.get("k", 3)),
        device=device
    )

    final_model = train_fn(
        model=final_model,
        u_train=u,
        x_train=x,
        epochs=epochs_final,
        lr=lr,
        batch_size=batch_size
    )

    # -------------------------------------------------
    # 4. Final test on M random pairs
    # -------------------------------------------------
    final_mse, final_rel, pred_cov = test_on_M_locations(
        final_model,
        data["u_test"].to(device),
        data["v_test"].to(device),
        data["true_cov"].to(device)
    )

    return {
        "model": final_model,
        "mse": final_mse,
        "rel": final_rel,
        "best_params": best_params,
        "tuning_df": tuning_df
    }

In [29]:
device = "cuda"

data = load_covnet_replicate(
    folder="Simulation/BM_rotated",
    repl=1,
    device=device
)

out = train_validation_tune_one_model(
    model_type="Shallow_KAN",
    data=data,
    candidate_grid=CN_SHALLOW_GRID,
    build_fn=build_KAN_model,
    train_fn=train_model,
    train_ratio=0.8,
    epochs_tune=1,
    epochs_final=200,
    lr=1e-2,
    batch_size=256,
    device=device,
    seed=12
)

print("Final RE:", out["rel"])
print(out["best_params"])
print(out["tuning_df"])


Tuning Shallow_KAN: {'R': 10}
checkpoint directory created: ./model
saving model version 0.0
Validation loss = 4.42996016e-03

Best params: {'R': 10.0, 'val_loss': 0.004429960157722235}
checkpoint directory created: ./model
saving model version 0.0
Final RE: 0.12455929815769196
{'R': 10.0, 'val_loss': 0.004429960157722235}
    R  val_loss
0  10   0.00443


In [30]:
out = train_validation_tune_one_model(
    model_type="Shallow_net",
    data=data,
    candidate_grid=CN_SHALLOW_GRID,
    build_fn=build_CN_model,
    train_fn=train_model,
    epochs_tune=1000,
    epochs_final=3000,
    lr=1e-2,
    batch_size=256,
    device=device
)


print("Final RE:", out["rel"])
print(out["best_params"])
print(out["tuning_df"])


Tuning Shallow_net: {'R': 10}
Validation loss = 9.03893146e-04

Best params: {'R': 10.0, 'val_loss': 0.0009038931457325816}
Final RE: 0.14364062249660492
{'R': 10.0, 'val_loss': 0.0009038931457325816}
    R  val_loss
0  10  0.000904


In [ ]:
pd.DataFrame(all_rows).to_csv("table1_running_results.csv", index=False)

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipykernel_721/4088889856.py", line 1, in <cell line: 0>
    pd.DataFrame(all_rows).to_csv("table1_running_results.csv", index=False)
  File "/usr/local/lib/python3.12/dist-packages/pandas/util/_decorators.py", line 333, in wrapper
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pandas/core/generic.py", line 3967, in to_csv
    return DataFrameRenderer(formatter).to_csv(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pandas/io/formats/format.py", line 1014, in to_csv
    csv_formatter.save()
  File "/usr/local/lib/python3.12/dist-packages/pandas/io/formats/csvs.py", line 251, in save
    with get_handle(
         ^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packag

In [ ]:
pd.DataFrame(all_rows)

,replicate,method,rel,excess
0,1,empirical,0.270474,NaN
1,1,BestSep,0.279600,NaN
2,1,Shallow_net,0.286884,0.031044
3,1,Deep_net,0.251435,0.002351
4,1,DeepShared_net,0.274576,0.029783
5,2,empirical,0.156625,NaN
6,2,BestSep,0.155795,NaN
7,2,Shallow_net,0.161049,0.008481
8,2,Deep_net,0.148292,0.006414
9,2,DeepShared_net,0.140406,0.010929


1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20
